# Code for formatting ANVIL-generated episode annotations

In [1]:
import pandas as pd

from analysis_helpers.constants import ANNOTATIONS_DIR

In [2]:
dtypes = {
    'Frame': 'Int64', 
    'Time': 'Float64', 
    'Scene info:narrative details - external': 'string', 
    'Scene info:narrative details - internal': 'string', 
    'Scene info:characters on screen': 'string', 
    'Scene info:Music Presence': 'string', 
    'speech:transcription': 'string', 
    'speech:character speaking': 'string', 
    'setting:indoor/outdoor': 'string', 
    'setting:setting': 'string', 
    'Scene name:scene name': 'string'
}
old_colnames = dtypes.keys()
new_colnames = (
    'Frame', 
    'Onset time', 
    'Narrative details (external events)', 
    'Narrative details (internal state)', 
    'Characters on screen', 
    'Music presence', 
    'Speech', 
    'Character speaking', 
    'Indoor/outdoor', 
    'Setting', 
    'Scene name'
)
replace_values = {
    'Narrative details (external events)': {'0': pd.NA},
    'Narrative details (internal state)': {'0' : pd.NA},
    'Characters on screen': {'0': pd.NA},
    'Music presence': {'0': 'no', '1': 'yes'},
    'Speech': {'0': pd.NA, '-1000': pd.NA},
    'Character speaking': {'0': pd.NA, '-1000': pd.NA},
    'Indoor/outdoor': {'1': 'indoor', '2': 'outdoor', '0': pd.NA, '-1000': pd.NA},
    'Setting': {'0': pd.NA, '-1000': pd.NA},
    'Scene name': {'-1000': pd.NA}
}

In [3]:
nd_external = new_colnames[2]
endframe_times = {}

for episode in ('atlep1', 'atlep2', 'arrdev'):
    raw_path = ANNOTATIONS_DIR.joinpath(f'{episode}-raw.tsv')
    df = pd.read_csv(raw_path, sep='\t', usecols=old_colnames, dtype=dtypes)
    # rename columns
    df.columns = new_colnames
    # get timestamp for last frame of video (full runtime)
    endframe_time = df['Onset time'].iloc[-1].item()
    # annotation files output by ANVIL contain current annotated values 
    # for every frame of video. Drop consecutive duplicates, keep only 
    # onset frame for each annotation.
    # Also account for occasional 2-6 frame delay between start of 
    # narrative details & speech annotation blocks
    df = df.loc[
        (df[nd_external].shift(fill_value='') != df[nd_external]) | 
        (df['Speech'].shift(fill_value='') != df['Speech'])
    ]
    df = df.loc[df[nd_external].shift(-1, fill_value='') != df[nd_external]]
    # videos included occasional 1-3s of black screen where commercial 
    # breaks would normally be, annotated as "BLACK SCREEN {1,2,3,...}".
    # Remove these rows and shift subsequent rows' onset times backward 
    # by their total duration
    black_screen_mask = df[nd_external].str.startswith('BLACK SCREEN')
    annot_durations = (
        df['Onset time'].shift(-1, fill_value=endframe_time) - df['Onset time']
    )
    black_screen_cumtime = annot_durations.where(black_screen_mask, 0).cumsum()
    df['Onset time'] -= black_screen_cumtime
    df = df.loc[~black_screen_mask]
    # clean up some floating point errors
    df['Onset time'] = df['Onset time'].round(2)
    black_screen_cumtime = black_screen_cumtime.round(2)
    endframe_times[episode] = endframe_time - black_screen_cumtime.iloc[-1].item()
    
    df.replace(replace_values, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_csv(raw_path.with_name(f'{episode}.csv'), index=False)

In [4]:
print('Timestamps of final video frame:')
for episode, time in endframe_times.items():
    print(f'    {episode}: {time}')

Timestamps of final video frame:
    atlep1: 1454.16
    atlep2: 1302.6
    arrdev: 1232.76
